# Generate a Triton repository on Colab CPU

Install, download a pinned checkpoint, inspect it and generate a repository.
No GPU or Docker needed. This example does not load the model or run inference.


In [ ]:
%pip install -q "triton-hf-builder==0.1.2" "huggingface-hub>=1.32,<2"

In [ ]:
from huggingface_hub import snapshot_download

source = snapshot_download(
    repo_id="HuggingFaceTB/SmolLM2-135M-Instruct",
    revision="12fd25f77366fa6b3b4b768ec3050bf629380bac",
    allow_patterns=[
        "config.json",
        "generation_config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.json",
        "merges.txt",
        "README.md",
        "LICENSE*",
        "NOTICE*",
    ],
)

In [ ]:
from triton_hf_builder import generate_repository, inspect_model

info = inspect_model(source)
info.to_dict()

In [ ]:
import tempfile
from pathlib import Path

# A new directory lets you rerun this cell without overwriting an earlier result.
output = Path(tempfile.mkdtemp(prefix="triton-hf-")) / "model_repository"
result = generate_repository(
    source=source,
    output=output,
    name="smollm2",
    runtime="transformers",
    image="nvcr.io/nvidia/tritonserver:26.06-py3",
    device="cpu",
)
print(result.repository)

In [ ]:
print((result.model_directory / "config.pbtxt").read_text())
result.manifest["validation"]